# StyleGAN2-ADA + Latent Diffusion — Geliştirilmiş Versiyon

In [ ]:
# Cell 1 - Latent Üretimi

!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
!pip install ninja -q

import sys
sys.path.insert(0, '/kaggle/working/stylegan2-ada-pytorch')

import torch, pickle, os, numpy as np, dnnlib, legacy
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

network_pkl = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl'
print('Model yükleniyor...')
with dnnlib.util.open_url(network_pkl) as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device)

num_samples = 20000
batch_size  = 100
all_w = []

print(f'{num_samples} latent vektör üretiliyor...')
with torch.no_grad():
    for i in tqdm(range(0, num_samples, batch_size)):
        curr_bs = min(batch_size, num_samples - i) 
        z = torch.randn([curr_bs, G.z_dim]).to(device)
        w = G.mapping(z, None)
        w = w[:, 0, :]  
        all_w.append(w.cpu().numpy())

all_w = np.concatenate(all_w, axis=0)
print(f'Shape: {all_w.shape}')  # (20000, 512)

os.makedirs('dataset', exist_ok=True)
np.save('dataset/latents_w_opt.npy', all_w)
print("Kaydedildi: dataset/latents_w_opt.npy")

In [ ]:
# Cell 2 - Classifier Eğitimi + Etiketleme

import torch, torch.nn as nn, torch.optim as optim
import numpy as np, pandas as pd, os
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights          
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score                    
from PIL import Image
from tqdm import tqdm

ATTRS_TO_USE    = ['Male', 'Eyeglasses', 'Smiling']
BATCH_SIZE      = 64
EPOCHS          = 5
LR              = 0.001
CELEBA_IMG_DIR  = '/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba'
CELEBA_ATTR_PATH = '/kaggle/input/celeba-dataset/list_attr_celeba.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_LIMIT = 30000


class CelebADataset(Dataset):
    def __init__(self, img_dir, attr_path, transform=None, limit=None):
        self.img_dir  = img_dir
        attr_df       = pd.read_csv(attr_path)
        if limit:
            attr_df = attr_df.iloc[:limit]
        self.filenames = attr_df['image_id'].values
        self.labels    = attr_df[ATTRS_TO_USE].replace(-1, 0).values
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        path  = os.path.join(self.img_dir, self.filenames[idx])
        image = Image.open(path).convert('RGB')
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        if self.transform:
            image = self.transform(image)
        return image, label


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = CelebADataset(CELEBA_IMG_DIR, CELEBA_ATTR_PATH, transform, limit=DATA_LIMIT)
train_size   = int(0.85 * len(full_dataset))
val_size     = len(full_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(f'Train: {train_size} | Val: {val_size}')

classifier    = models.resnet18(weights=ResNet18_Weights.DEFAULT)
classifier.fc = nn.Linear(classifier.fc.in_features, len(ATTRS_TO_USE))
classifier    = classifier.to(DEVICE)
criterion     = nn.BCEWithLogitsLoss()
optimizer_cls = optim.Adam(classifier.parameters(), lr=LR)

if os.path.exists('classifier_resnet18.pth'):
    print('Kaydedilmiş model yükleniyor...')
    classifier.load_state_dict(torch.load('classifier_resnet18.pth'))
else:
    print('Sınıflandırıcı eğitimi başlıyor...')
    for epoch in range(EPOCHS):
        classifier.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer_cls.zero_grad()
            loss = criterion(classifier(images), labels)
            loss.backward()
            optimizer_cls.step()
            running_loss += loss.item()

        classifier.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                preds  = (torch.sigmoid(classifier(images)) > 0.5).float()
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.numpy())
        all_preds  = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        f1s = f1_score(all_labels, all_preds, average=None, zero_division=0)
        print(f'Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f}', end='')
        for attr, f1 in zip(ATTRS_TO_USE, f1s):
            print(f' | {attr}: F1={f1:.3f}', end='')
        print()

    torch.save(classifier.state_dict(), 'classifier_resnet18.pth')
    print('Classifier eğitildi ve kaydedildi.')

# Etiketleme
print('Latent vektörler etiketleniyor...')
classifier.eval()
w_vectors = np.load('dataset/latents_w_opt.npy')
w_tensor  = torch.tensor(w_vectors, dtype=torch.float32).to(DEVICE)
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
generated_labels = []

with torch.no_grad():
    for i in tqdm(range(0, len(w_tensor), 32)):
        batch_w = w_tensor[i : i+32]
        bw_3d   = batch_w.unsqueeze(1).repeat(1, G.num_ws, 1)  # [B,18,512]
        imgs    = G.synthesis(bw_3d, noise_mode='const')
        imgs    = torch.clamp((imgs + 1) * 0.5, 0, 1)
        imgs    = torch.nn.functional.interpolate(imgs, (224, 224), mode='bilinear', align_corners=False)
        imgs    = normalize(imgs)
        preds   = (torch.sigmoid(classifier(imgs)) > 0.5).float()
        generated_labels.append(preds.cpu().numpy())

generated_labels = np.concatenate(generated_labels, axis=0)
print(f'Etiketleme bitti. Shape: {generated_labels.shape}')
np.save('dataset/labels.npy', generated_labels)

In [ ]:
# Cell 3 - Diffusion Model Eğitimi

import torch, torch.nn as nn, torch.optim as optim
import numpy as np, math, os, copy
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm

BATCH_SIZE        = 64
LR                = 1e-4
EPOCHS            = 150
DEVICE            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
W_DIM             = 512
NUM_CLASSES       = 3
TIMESTEPS         = 1000
CFG_DROPOUT_PROB  = 0.1
EMA_DECAY         = 0.9999

# Veri yükleme 
print('Veriler okunuyor...')
try:                                                    
    w_vectors = np.load('dataset/latents_w_opt.npy')
except FileNotFoundError:
    w_vectors = np.load('dataset/latents_w.npy')

if w_vectors.ndim == 3:
    w_vectors = w_vectors[:, 0, :]

labels       = np.load('dataset/labels.npy')
w_tensor     = torch.tensor(w_vectors, dtype=torch.float32)
lbl_tensor   = torch.tensor(labels,    dtype=torch.float32)
full_ds      = TensorDataset(w_tensor, lbl_tensor)
train_n      = int(0.9 * len(full_ds))
val_n        = len(full_ds) - train_n
train_ds, val_ds = random_split(full_ds, [train_n, val_n])
dataloader   = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
print(f'Train: {train_n} | Val: {val_n}')


# Model
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        half_dim   = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=time.device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class LatentDiffusionMLP(nn.Module):
    def __init__(self, w_dim=512, num_classes=3, hidden_dim=1024):
        super().__init__()
        self.time_mlp   = nn.Sequential(
            SinusoidalPositionEmbeddings(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.cond_mlp   = nn.Sequential(
            nn.Linear(num_classes, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.input_proj = nn.Linear(w_dim, hidden_dim)
        self.block1     = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim))
        self.block2     = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim))
        self.block3     = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim))
        self.output_proj = nn.Linear(hidden_dim, w_dim)
        self.act        = nn.SiLU()

    def forward(self, x, t, c):
        h = self.input_proj(x) + self.time_mlp(t) + self.cond_mlp(c)
        h = h + self.block1(self.act(h))
        h = h + self.block2(self.act(h))
        h = h + self.block3(self.act(h))
        return self.output_proj(self.act(h))


class DiffusionManager:
    def __init__(self, timesteps=1000, beta_start=0.0001, beta_end=0.02, device='cuda'):
        self.timesteps = timesteps
        self.device    = device
        betas          = torch.linspace(beta_start, beta_end, timesteps).to(device)
        alphas         = 1. - betas
        ac             = torch.cumprod(alphas, dim=0)
        self.sqrt_ac   = torch.sqrt(ac)
        self.sqrt_1mac = torch.sqrt(1. - ac)

    def add_noise(self, x_start, t):
        noise = torch.randn_like(x_start)
        x_t   = self.sqrt_ac[t][:, None] * x_start + self.sqrt_1mac[t][:, None] * noise
        return x_t, noise

class EMA:
    def __init__(self, model, decay=0.9999):
        self.model = copy.deepcopy(model).eval()
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        for ema_p, p in zip(self.model.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(p.data, alpha=1 - self.decay)


# Eğitim
model = LatentDiffusionMLP(W_DIM, NUM_CLASSES, hidden_dim=2048).to(DEVICE)
diffusion = DiffusionManager(TIMESTEPS, device=DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
ema       = EMA(model, decay=EMA_DECAY)                               
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, eta_min=1e-6)

print('Difüzyon modeli eğitimi başlıyor...')
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for batch_w, batch_labels in dataloader:
        batch_w, batch_labels = batch_w.to(DEVICE), batch_labels.to(DEVICE)
        t = torch.randint(0, TIMESTEPS, (batch_w.shape[0],), device=DEVICE).long()
        noisy_w, noise = diffusion.add_noise(batch_w, t)

        # CFG dropout: %10 ihtimalle null token (-1)
        if np.random.random() < CFG_DROPOUT_PROB:
            batch_labels = torch.ones_like(batch_labels) * -1

        optimizer.zero_grad()
        predicted_noise = model(noisy_w, t, batch_labels)
        loss = criterion(predicted_noise, noise)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  
        optimizer.step()
        ema.update(model)                                                
        epoch_loss += loss.item()

    scheduler.step()                                                   

    if (epoch + 1) % 10 == 0:
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_w, batch_labels in val_loader:
                batch_w, batch_labels = batch_w.to(DEVICE), batch_labels.to(DEVICE)
                t = torch.randint(0, TIMESTEPS, (batch_w.shape[0],), device=DEVICE).long()
                noisy_w, noise = diffusion.add_noise(batch_w, t)
                pred = model(noisy_w, t, batch_labels)
                val_loss += criterion(pred, noise).item()
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
              f'Train: {epoch_loss/len(dataloader):.5f} | '
              f'Val: {val_loss/len(val_loader):.5f} | '
              f'LR: {scheduler.get_last_lr()[0]:.2e}')

torch.save(model.state_dict(),     'latent_diffusion_mlp.pth')
torch.save(ema.model.state_dict(), 'latent_diffusion_mlp_ema.pth') 
print('Model ve EMA model kaydedildi.')

In [ ]:
# Cell 4 - Yön Hesabı ve Manipülasyon

import torch, numpy as np, matplotlib.pyplot as plt

MY_SEED  = 42
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ema_model = LatentDiffusionMLP(W_DIM, NUM_CLASSES, hidden_dim=2048).to(DEVICE)
ema_model.load_state_dict(torch.load('latent_diffusion_mlp_ema.pth'))
ema_model.eval()
print('EMA model yüklendi.')


def get_direction_cfg(
    model, diffusion, attr_idx,
    seed=42,
    t_values=[100, 300, 500, 700, 900],  
    guidance_scale=3.0,                  
    samples_per_t=5,
):
    torch.manual_seed(seed)
    all_deltas = []

    # Null token: eğitimde -1 kullanıldı
    c_null = (torch.ones(1, NUM_CLASSES) * -1).to(DEVICE)
    c_pos  = torch.zeros(1, NUM_CLASSES).to(DEVICE)
    c_neg  = torch.zeros(1, NUM_CLASSES).to(DEVICE)
    c_pos[0, attr_idx] = 1.0
    c_neg[0, attr_idx] = 0.0

    with torch.no_grad():
        for t_val in t_values:                                 
            for _ in range(samples_per_t):
                noise = torch.randn(1, W_DIM).to(DEVICE)
                t     = torch.tensor([t_val], device=DEVICE).long()

                eps_uncond = model(noise, t, c_null)             
                eps_pos    = model(noise, t, c_pos)
                eps_neg    = model(noise, t, c_neg)

                # CFG: unconditional + scale * (conditional - unconditional)
                guided_pos = eps_uncond + guidance_scale * (eps_pos - eps_uncond)
                guided_neg = eps_uncond + guidance_scale * (eps_neg - eps_uncond)

                # neg - pos: pozitif attribute yönüne doğru
                delta = guided_neg - guided_pos
                all_deltas.append(delta)

    avg_delta = torch.stack(all_deltas).mean(dim=0)
    return avg_delta / torch.norm(avg_delta)


# Yönü hesapla
ATTR_INDEX = 1
print(f'Yön hesaplanıyor: {ATTRS_TO_USE[ATTR_INDEX]}...')
direction = get_direction_cfg(ema_model, diffusion, ATTR_INDEX, seed=MY_SEED)

# Kaynak W seç
try:
    all_ws = np.load('dataset/latents_w_opt.npy')
except FileNotFoundError:
    all_ws = np.load('dataset/latents_w.npy')
    if all_ws.ndim == 3:
        all_ws = all_ws[:, 0, :]

IDX      = 150
source_w = torch.tensor(all_ws[IDX]).unsqueeze(0).to(DEVICE)


def w_to_img(w_1d):
    #[1, 512] W vektörünü PIL görüntüsüne çevirir.
    with torch.no_grad():
        w3d = w_1d.unsqueeze(1).repeat(1, 18, 1)
        img = G.synthesis(w3d, noise_mode='const')
    return (img.permute(0,2,3,1)*127.5+128).clamp(0,255).to(torch.uint8).cpu().numpy()[0]

scales = [-8, -5, 0, 5, 8]
fig, axes = plt.subplots(1, len(scales), figsize=(20, 5))
for ax, scale in zip(axes, scales):
    edited_w = source_w + scale * direction
    ax.imshow(w_to_img(edited_w))
    ax.set_title(f'scale={scale}', fontsize=12)
    ax.axis('off')
attr_name = ATTRS_TO_USE[ATTR_INDEX]
plt.suptitle(f'CFG Direction Sweep — {attr_name} | Seed: {MY_SEED}', fontsize=13)
plt.tight_layout()
plt.show()